In [14]:
from pathlib import Path
import sys
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().parent

sys.path.append(
    str(PROJECT_ROOT)
)

from src.models import Siamese2D
from src.losses import contrastive_loss

In [15]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

cpu


In [16]:
IMAGE_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "images"
    / "gasf"
)

model = Siamese2D(
    embedding_dim=128
).to(DEVICE)

In [17]:
classes = ["normal", "irf", "orf"]

NORMALIZED_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "normalized"
)

X_list = []
y_list = []

for label, class_name in enumerate(classes):

    X_class = np.loadtxt(
        NORMALIZED_DIR / f"{class_name}.csv",
        delimiter=","
    ).astype(np.float32)

    X_list.append(X_class)
    y_list.extend(
        [label] * len(X_class)
    )

X = np.concatenate(X_list, axis=0)
y = np.asarray(y_list)

split = np.load(
    PROJECT_ROOT
    / "results"
    / "fsl_split.npz"
)

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

print("X:", X.shape)
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

X: (300, 100000)
Train: 210
Validation: 45
Test: 45


In [18]:
from src.image_dataset import (
    SiameseImagePairDataset
)

train_dataset = SiameseImagePairDataset(
    IMAGE_DIR,
    train_idx,
    y,
    classes
)

val_dataset = SiameseImagePairDataset(
    IMAGE_DIR,
    val_idx,
    y,
    classes
)

In [19]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [20]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [21]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    device
):

    model.train()
    total_loss = 0.0

    for x1, x2, label in loader:

        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)

        optimizer.zero_grad()

        z1, z2 = model(x1, x2)
        loss = contrastive_loss(z1, z2, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(
    model,
    loader,
    device
):

    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for x1, x2, label in loader:

            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)

            z1, z2 = model(x1, x2)
            loss = contrastive_loss(z1, z2, label)
            total_loss += loss.item()

    return total_loss / len(loader)


EPOCHS = 20
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = MODEL_DIR / "fsl_gasf_best.pt"
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        DEVICE
    )

    val_loss = validate(
        model,
        val_loader,
        DEVICE
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.6f} | "
        f"Val loss: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            },
            CHECKPOINT_PATH
        )

        print("Checkpoint saved:", CHECKPOINT_PATH)

c:\Users\asus\OneDrive\Desktop\siamese-network\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 01/20 | Train loss: 0.495772 | Val loss: 0.486448
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gasf_best.pt
Epoch 02/20 | Train loss: 0.492561 | Val loss: 0.486925
Epoch 03/20 | Train loss: 0.491182 | Val loss: 0.487239
Epoch 04/20 | Train loss: 0.488742 | Val loss: 0.486409
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gasf_best.pt
Epoch 05/20 | Train loss: 0.485746 | Val loss: 0.482291
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gasf_best.pt
Epoch 06/20 | Train loss: 0.481879 | Val loss: 0.482093
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gasf_best.pt
Epoch 07/20 | Train loss: 0.474427 | Val loss: 0.492621
Epoch 08/20 | Train loss: 0.469455 | Val loss: 0.494279
Epoch 09/20 | Train loss: 0.469857 | Val loss: 0.467514
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gasf_best.pt
Epoch 10/20 | Train loss: 0.466999 | Val loss: 0.46

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = MODEL_DIR / "fsl_gasf_best.pt"

if "epoch" in globals() and "val_loss" in globals():

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss
        },
        CHECKPOINT_PATH
    )

    print("Checkpoint saved:", CHECKPOINT_PATH)

else:

    print(
        "Checkpoint not saved: run a training epoch first."
    )

Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gasf_best.pt


: 